# 🏟️ Multi-Object Detection & Persistent ID Tracking

This notebook walks through the complete pipeline step-by-step:
1. Download a public sports video
2. Detect players with YOLOv8
3. Track them with ByteTrack
4. Visualize results (annotated video, heatmap, count chart, trajectories)

**Video Source:** [FIFA World Cup 2022 Match Highlights](https://www.youtube.com/watch?v=l7KpNAa3NhA)

## 0. Setup & Imports

In [ ]:
import os
import sys
import logging

# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath('..'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
print('✅ Imports ready')

## 1. Download Video

In [ ]:
from src.downloader import download_video

# Public football highlight reel (FIFA World Cup 2022)
VIDEO_URL = "https://www.youtube.com/watch?v=l7KpNAa3NhA"
VIDEO_PATH = "../input_video.mp4"

video_path = download_video(
    url=VIDEO_URL,
    output_path=VIDEO_PATH,
    max_duration_sec=120,   # 2-minute clip for quick demo
)
print(f'Video ready: {video_path}')

## 2. Inspect a Single Frame — Detection Test

In [ ]:
import cv2
import supervision as sv
import matplotlib.pyplot as plt

from src.detector import PersonDetector

detector = PersonDetector(model_path='yolov8m.pt', confidence=0.35)

# Grab a representative frame
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 120)  # frame 120 (~4s in)
ret, frame = cap.read()
cap.release()

# Detect
detections = detector.detect(frame)
print(f'Detected {len(detections)} players in frame')

# Annotate
annotated = sv.BoxAnnotator().annotate(frame.copy(), detections)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'Detection result — {len(detections)} players detected', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Run the Full Pipeline

In [ ]:
from src.pipeline import run_pipeline

OUTPUT_PATH = '../outputs/annotated_output.mp4'
ANALYTICS_DIR = '../outputs'

run_pipeline(
    input_path=video_path,
    output_path=OUTPUT_PATH,
    model_path='yolov8m.pt',
    conf=0.35,
    frame_skip=1,          # process every frame
    trace_length=45,
    device='cpu',
    save_analytics=True,
    analytics_dir=ANALYTICS_DIR,
)
print('✅ Pipeline complete!')

## 4. View Analytics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

analytics_files = {
    'Movement Heatmap': '../outputs/heatmap.png',
    'Player Count Over Time': '../outputs/count_over_time.png',
    'Player Trajectories': '../outputs/trajectories.png',
}

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, (title, path) in zip(axes, analytics_files.items()):
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=13, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Not generated yet', ha='center', va='center')
    ax.axis('off')

plt.suptitle('Tracking Analytics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Play Annotated Video (inline)

In [ ]:
from IPython.display import Video
Video(OUTPUT_PATH, width=900)

## 6. Summary Statistics

In [ ]:
print('Pipeline complete!')
print('='*50)
print(f'  Annotated video : {OUTPUT_PATH}')
print(f'  Heatmap         : {ANALYTICS_DIR}/heatmap.png')
print(f'  Count chart     : {ANALYTICS_DIR}/count_over_time.png')
print(f'  Trajectories    : {ANALYTICS_DIR}/trajectories.png')
print('='*50)